In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

pd.set_option('display.max_columns', 30)

target_col = 'diabetes_stage_encoded'
labels_map = {0: 'No Diabetes', 1: 'Pre-Diabetes', 2: 'Gestational', 3: 'Type 2'}

train_full = pd.read_csv('../for progress/results/outputs/stage6_train_balanced.csv')
test_df   = pd.read_csv('../for progress/results/outputs/stage6_test_holdout.csv')

SAMPLE_PER_CLASS = 4000
parts = []
for cls, g in train_full.groupby(target_col):
    parts.append(g.sample(min(len(g), SAMPLE_PER_CLASS), random_state=42))
train_df = pd.concat(parts, ignore_index=True)

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]
X_test  = test_df.drop(columns=[target_col])
y_test  = test_df[target_col]

def evaluate(name, model, X_tr, y_tr, X_te, y_te, results):
    y_pred = model.predict(X_te)
    report = classification_report(y_te, y_pred, target_names=[labels_map[c] for c in sorted(labels_map)],
                                    output_dict=True, zero_division=0)
    acc = report['accuracy']
    f1w = report['weighted avg']['f1-score']
    results.append({'variant': name, 'accuracy': acc, 'weighted_f1': f1w})
    print(f"--- {name} ---")
    print(classification_report(y_te, y_pred, target_names=[labels_map[c] for c in sorted(labels_map)], zero_division=0))
    return y_pred

def plot_confusion(y_te, y_pred, title):
    cm = confusion_matrix(y_te, y_pred, labels=sorted(labels_map))
    disp = ConfusionMatrixDisplay(cm, display_labels=[labels_map[c] for c in sorted(labels_map)])
    fig, ax = plt.subplots(figsize=(5.5, 5))
    disp.plot(ax=ax, cmap='Blues', colorbar=False, xticks_rotation=45)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

results = []

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'max_features': ['sqrt', 'log2'],
}

grid = GridSearchCV(RandomForestClassifier(random_state=42, n_jobs=-1), param_grid,
                     scoring='f1_weighted', cv=3, n_jobs=-1)
grid.fit(X_train, y_train)

rf_tuned = grid.best_estimator_


In [2]:
#use tkinder to create a simple GUI for user input and predict sample data and fill random data

import tkinter as tk
from tkinter import messagebox



In [9]:
# The model must already be trained as rf_tuned in the notebook.

feature_names = list(X_train.columns)

window = tk.Tk()
window.title("Diabetes Stage Prediction")
window.resizable(False, False)

frame = tk.Frame(window, padx=15, pady=15)
frame.pack()

input_frame = tk.LabelFrame(frame, text="Patient Features", padx=10, pady=10)
input_frame.grid(row=0, column=0, padx=10, pady=10)

entries = {}

for row, feature in enumerate(feature_names):
    label = tk.Label(input_frame, text=feature.replace("_", " ").title())
    label.grid(row=row, column=0, sticky="w", padx=5, pady=3)

    entry = tk.Entry(input_frame, width=20)
    entry.insert(0, "0")
    entry.grid(row=row, column=1, padx=5, pady=3)

    entries[feature] = entry


result_label = tk.Label(
    frame,
    text="Enter feature values and click Predict",
    font=("Arial", 11),
    wraplength=350
)
result_label.grid(row=2, column=0, pady=10)


def submit_data():
    try:
        values = {
            feature: float(entries[feature].get())
            for feature in feature_names
        }

        user_data = pd.DataFrame([values], columns=feature_names)

        prediction = rf_tuned.predict(user_data)[0]
        probabilities = rf_tuned.predict_proba(user_data)[0]
        confidence = probabilities.max() * 100

        result_label.config(
            text=(
                f"Predicted stage: {labels_map[prediction]}\n"
                f"Confidence: {confidence:.2f}%"
            ),
            fg="darkblue"
        )

    except ValueError:
        messagebox.showerror(
            "Invalid input",
            "Please enter numeric values for every feature."
        )


predict_button = tk.Button(
    frame,
    text="Predict Diabetes Stage",
    command=submit_data,
    bg="green",
    fg="white",
    padx=10,
    pady=5
)
predict_button.grid(row=1, column=0, pady=10)

#random data generator button to fill the input fields with random values for testing with all type predictions
def generate_random_data():
    random_values = {
        feature: np.random.uniform(
            low=X_train[feature].min(),
            high=X_train[feature].max()
        )
        for feature in feature_names
    }

    for feature, value in random_values.items():
        entries[feature].delete(0, tk.END)
        entries[feature].insert(0, f"{value:.2f}")

random_button = tk.Button(
    frame,
    text="Generate Random Data",
    command=generate_random_data,
    bg="yellow",
    fg="black",
    padx=10,
    pady=5
)
random_button.grid(row=1, column=1, pady=10)

window.mainloop()